# sub-16: are the repeated directions real? + impact on the OD

**Context** (`sub16_disputed_sweeps.ipynb`):
- The criteria S1 (frame evidence), S2 (pRF model) and S4 (travelling-wave latency) rank the truth 1st in 11-12/12 tests. But these tests only compared chains with the generator structure (each direction once).
- On the suspect runs, the **free chains** win. All three lack 45+ and repeat 0+ and 135-. Two explanations:
  - a reading bias (45+ read as 0+, already seen in validation of the free search);
  - the pilot was not a permutation of the 8 directions.

**Step 1 — validation with repeats allowed** (known runs; decoders never trained on the tested sequence; sub-05 RE01 = same configuration as the suspect runs):
- **single sweep**: at each sweep of a known run, the 8 directions are tried (everything else fixed). Accuracy and confusion matrix of each criterion: are 45+ -> 0+ errors made?
- **triple**: 3 hidden sweeps, 512 combinations with repeats. Rank of the truth.

**Step 2 — suspect runs**: same single-sweep test on the free chain and on the chain with structure. A repeated direction is confirmed if the validated criteria choose it against the 7 others, with a margin larger than any error made in validation.

**Step 3 — impact on the OD of sub-16** (V1, V2, V3) with each design: without the 3 runs, current analysis (A everywhere), free chains, chains with structure. If the OD barely changes between the recovered designs, the remaining uncertainty does not matter for the paper.

Read only.

In [1]:
import os, glob, math
from functools import lru_cache
import itertools
import numpy as np, pandas as pd, nibabel as nb
import scipy.io as sio
import matplotlib.pyplot as plt

main = '/Users/uriel/disks/meso_shared/amblyo7T_prf'
pp = f'{main}/derivatives/pp_data'
out_dir = '/Users/uriel/Desktop/fig_od/sub16_combined'
os.makedirs(out_dir, exist_ok=True)
TR = 2.0
lags = [1, 2, 3, 4]                       # decoder: data at t+lag decodes the stimulus at t
model_lags = [2, 3]                       # pRF evidence: response pattern at t+lag
alphas = [1e1, 1e2, 1e3, 1e4, 1e5, 1e6]
dec_rois, dec_nmax = [1, 2, 3], 3000      # best variant of sub16_stimulus_decoder.ipynb
RW = ['RingsLeftEye_run-01', 'WedgesLeftEye_run-01', 'RingsRightEye_run-01', 'WedgesRightEye_run-01']
SUSPECTS = ['BarsLeftEye_run-01', 'BarsRightEye_run-01', 'BarsRightEye_run-02']
rng = np.random.default_rng(0)


/var/folders/60/q71xm4hx45x3db5d6mzl8r4h0000gn/T/ipykernel_7630/2562092748.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import numpy as np, pandas as pd, nibabel as nb


## Functions (same as the previous notebooks)

In [2]:
def build_vdm(mat_fn):
    """Same conversion as create_vdm_files.ipynb (50 x 50, transposed, padded)."""
    im = np.asarray(sio.loadmat(mat_fn, squeeze_me=True, struct_as_record=False)['s'].im, float)
    out = np.zeros((50, 50, im.shape[-1]))
    ii = np.floor(np.arange(50) * 102 / 50).astype(int)
    for t in range(im.shape[-1]):
        f = (im[..., t] * 255).astype(np.uint8).T
        sq = np.vstack([np.zeros((12, 102), np.uint8), f, np.zeros((13, 102), np.uint8)])
        out[..., t] = (sq > 5)[np.ix_(ii, ii)]
    return out

vdm = {eye: np.load(f'{main}/derivatives/vdm/task-BarsBarsRingsWedges{eye}_vdm.npy') for eye in ['LeftEye', 'RightEye']}
A = vdm['LeftEye'][..., :120]
B = build_vdm(f'{main}/sourcedata/C/jp/cond-1-stim-3.mat')
true_stim = {'A': A, 'B': B}


def run_stim(r, kind=None):
    """Known stimulus (50,50,120) of a run."""
    eye = 'LeftEye' if 'LeftEye' in r else 'RightEye'
    if r.startswith('Rings'): return vdm[eye][..., 240:360]
    if r.startswith('Wedges'): return vdm[eye][..., 360:480]
    return true_stim[kind]


def ds(st):
    """(50,50,T) -> (T, 625), 2x2 average."""
    return st.reshape(25, 2, 25, 2, -1).mean((1, 3)).reshape(625, -1).T

rr, cc = np.meshgrid(np.arange(50) - 24.5, np.arange(50) - 24.5, indexing='ij')
X_pix, Y_pix = cc, -rr


def frame_label(f):
    m = f > 0
    if m.sum() < 3: return None, np.nan
    best = None
    for a in [0, 45, 90, 135]:
        th = np.deg2rad(a); p = X_pix[m] * np.cos(th) + Y_pix[m] * np.sin(th)
        if best is None or p.std() < best[0]: best = (p.std(), a, p.mean())
    return best[1], best[2]


def segment(stim):
    lab = [frame_label(stim[..., t]) for t in range(stim.shape[-1])]
    ev, t = [], 0
    while t < len(lab):
        o, p = lab[t]; u = t + 1
        if o is None:
            while u < len(lab) and lab[u][0] is None: u += 1
            ev.append(('b', t, u - t))
        else:
            while u < len(lab) and lab[u][0] == o and u - t < 12: u += 1
            ev.append((f"{o}{'+' if lab[u - 1][1] > p else '-'}", t, u - t))
        t = u
    return ev

templates = {}
for stim in [A, B]:
    for k, s, l in segment(stim):
        if k != 'b' and l == 12 and k not in templates:
            templates[k] = stim[..., s:s + 12]
print(sorted(templates))


def tmpl(k, L):
    """Sweep k resampled to L frames, full resolution (50,50,L)."""
    return templates[k][..., np.round(np.linspace(0, 11, L)).astype(int)]


def chain_to_stim(chain):
    st = np.zeros((50, 50, 120))
    for k, t0, L in chain:
        if k == 'b': continue
        fr = tmpl(k, L)
        n = min(L, 120 - t0)
        st[..., t0:t0 + n] = fr[..., :n]
    return st


def chain_str(chain):
    return ' '.join(f'b{L}' if k == 'b' else (k if L == 12 else f'{k}/{L}') for k, t0, L in chain)

true_chain = {k: [(e[0], e[1], e[2]) for e in segment(v)] for k, v in true_stim.items()}
print('A:', chain_str(true_chain['A'])); print('B:', chain_str(true_chain['B']))

['0+', '0-', '135+', '135-', '45+', '45-', '90+', '90-']
A: b6 135+ 90+ b5 45- 0+ 135- 0- 45+ 90- 135- 135-/1
B: 0+ 135- 90- 45- 0- 135+ 90+ 45+ 0+ 135-


In [3]:
cache = {}
def load_subject(s):
    if s in cache: return cache[s]
    roi = np.concatenate([nb.load(glob.glob(f'{pp}/{s}/fsnative/rois/{s}_hemi-{h}_fmriprep_dct_*_rois-group-mmp.func.gii')[0]).darrays[0].data
                          for h in ['L', 'R']])
    prf = {eye: np.hstack([np.vstack([a.data for a in nb.load(
        f'{pp}/{s}/fsnative/prfmodel/prf_derivatives/{s}_task-RingsWedges{eye}_hemi-{h}_fmriprep_dct_psc_concat_prf-css_deriv.func.gii').darrays])
        for h in ['L', 'R']]) for eye in ['LeftEye', 'RightEye']}
    cache[s] = (roi, prf)
    return cache[s]

run_cache = {}
def load_run(s, r):
    if (s, r) in run_cache: return run_cache[(s, r)]
    hs = []
    for h in ['L', 'R']:
        d = np.vstack([a.data for a in nb.load(
            f'{pp}/{s}/fsnative/func/fmriprep_dct_psc/{s}_ses-01_task-{r}_hemi-{h}_space-fsnative_fmriprep_dct_psc_bold.func.gii').darrays])
        hs.append(d if d.shape[0] == 120 else d.T)
    run_cache[(s, r)] = np.hstack(hs).astype(np.float32)
    return run_cache[(s, r)]


def select_vertices(s, rois, n_max):
    roi, prf = load_subject(s)
    r2 = (prf['LeftEye'][0] + prf['RightEye'][0]) / 2
    ok = np.isfinite(r2) & (r2 > 0.1) & ((roi > 0) if rois is None else np.isin(roi, rois))
    idx = np.where(ok)[0]
    return idx[np.argsort(-r2[idx])[:n_max]]


def zc(a, axis=0):
    a = a - a.mean(axis, keepdims=True)
    return a / (a.std(axis, keepdims=True) + 1e-9)


def features(y):
    """(120, nv) -> (120, nv * n_lags): row t = data at t+lag."""
    y = zc(np.nan_to_num(y))
    cols = []
    for l in lags:
        z = np.zeros_like(y); z[:120 - l] = y[l:]
        cols.append(z)
    return np.hstack(cols) / np.sqrt(y.shape[1] * len(lags))

In [4]:
def frame_corr(D, S):
    """Median over non-blank frames of the correlation between decoded and true frames."""
    m = S.sum(1) > 0
    Dc = D[m] - D[m].mean(1, keepdims=True); Sc = S[m] - S[m].mean(1, keepdims=True)
    return float(np.median((Dc * Sc).sum(1) / (np.sqrt((Dc ** 2).sum(1) * (Sc ** 2).sum(1)) + 1e-9)))


def fit_predict(Xtr, Ytr, Xte, alpha):
    mu = Ytr.mean(0)
    K = Xtr @ Xtr.T
    W = np.linalg.solve(K + alpha * np.eye(len(K)), Ytr - mu)
    return Xte @ Xtr.T @ W + mu


def train_decoder(train):
    """train: list of (X, Y). Returns predict(X), chosen alpha, gain, LORO frame corr."""
    best = None
    for a in alphas:
        preds = [fit_predict(np.vstack([t[0] for j, t in enumerate(train) if j != i]),
                             np.vstack([t[1] for j, t in enumerate(train) if j != i]), train[i][0], a) for i in range(len(train))]
        sc = np.mean([frame_corr(p, t[1]) for p, t in zip(preds, train)])
        if best is None or sc > best[0]:
            best = (sc, a, preds)
    sc, a, preds = best
    P, Y = np.vstack(preds), np.vstack([t[1] for t in train])
    mu = Y.mean(0)
    gain = float(((P - mu) * (Y - mu)).sum() / ((Y - mu) ** 2).sum())   # shrinkage of the decoded amplitude
    Xtr, Ytr = np.vstack([t[0] for t in train]), np.vstack([t[1] for t in train])
    return (lambda X: fit_predict(Xtr, Ytr, X, a)), a, gain, sc, mu

In [5]:
pix = 2 * math.degrees(math.atan(50.6 / 2 / 242)) / 50
Xd, Yd = X_pix * pix, Y_pix * pix
t_ = np.arange(0, 32, TR)
g = lambda a: t_ ** (a - 1) * np.exp(-t_) / math.gamma(a)
H = g(6) - g(16) / 6; H /= H.sum()
N_fft = 256
Hf = np.fft.rfft(H, n=N_fft)


def conv_hrf(d):
    return np.fft.irfft(np.fft.rfft(d, n=N_fft, axis=0) * Hf[:, None], n=N_fft, axis=0)[:d.shape[0]]


def prf_score(stim, s, r):
    """Median correlation (V1-V3 vertices, pRFs of the eye) between data and pRF prediction, best delay -2..+2 TR."""
    roi, prf = load_subject(s)
    D = prf['LeftEye' if 'LeftEye' in r else 'RightEye']
    y = load_run(s, r)
    idx = np.where((D[0] > 0.2) & (D[1] < 6) & np.isin(roi, [1, 2, 3]) & np.isfinite(y).all(0) & (y.std(0) > 0))[0]
    x0, y0, sd, n = D[7][idx], D[8][idx], np.clip(D[4][idx], 0.2, None), np.clip(D[11][idx], 0.05, 1)
    G = np.exp(-((Xd.ravel()[:, None] - x0) ** 2 + (Yd.ravel()[:, None] - y0) ** 2) / (2 * sd ** 2))
    p = conv_hrf(np.clip(np.moveaxis(stim, -1, 0).reshape(120, -1) @ G, 0, None) ** n)
    yy = y[:, idx]
    best = -np.inf
    for sh in range(-2, 3):
        q = np.roll(p, sh, 0)
        if sh > 0: q[:sh] = p[0]
        elif sh < 0: q[sh:] = p[-1]
        best = max(best, float(np.median((zc(q) * zc(yy)).mean(0))))
    return best

In [6]:
keys = sorted(templates)
T = 120


def decoder_scores(D, gain, mu):
    """Frame scores from the decoded movie, relative to a blank frame: dict k -> (T, 12)."""
    Dc = D - mu
    blank_err = ((Dc + gain * mu) ** 2).sum(1)
    out = {}
    for k in keys:
        F = ds(tmpl(k, 12))
        err = ((Dc[:, None, :] - gain * (F - mu)[None]) ** 2).sum(2)
        out[k] = blank_err[:, None] - err
    return out


model_cache = {}
def model_scores(s, r):
    """Frame scores from the pRF model: spatial correlation between the response pattern at t+lag and the pRF drive of each template frame."""
    if (s, r) in model_cache: return model_cache[(s, r)]
    roi, prf = load_subject(s)
    Dp = prf['LeftEye' if 'LeftEye' in r else 'RightEye']
    y = load_run(s, r)
    idx = np.where((Dp[0] > 0.2) & (Dp[1] < 6) & np.isin(roi, [1, 2, 3]) & np.isfinite(y).all(0) & (y.std(0) > 0))[0]
    x0, y0, sd, n = Dp[7][idx], Dp[8][idx], np.clip(Dp[4][idx], 0.2, None), np.clip(Dp[11][idx], 0.05, 1)
    G = np.exp(-((Xd.ravel()[:, None] - x0) ** 2 + (Yd.ravel()[:, None] - y0) ** 2) / (2 * sd ** 2))
    Y = zc(y[:, idx])
    out = {}
    for k in keys:
        drv = np.clip(np.moveaxis(tmpl(k, 12), -1, 0).reshape(12, -1) @ G, 0, None) ** n
        Dz = zc(drv, axis=1)
        acc = 0
        for l in model_lags:
            Yl = np.zeros_like(Y); Yl[:T - l] = Y[l:]
            acc = acc + zc(Yl, axis=1) @ Dz.T / len(idx)
        out[k] = acc / len(model_lags)
    model_cache[(s, r)] = out
    return out


def normalise(S, center):
    allv = np.concatenate([S[k].ravel() for k in keys])
    c = np.median(allv) if center else 0.0
    sd = allv.std() + 1e-12
    return {k: (S[k] - c) / sd for k in keys}


def combine(*Ss):
    return {k: sum(S[k] for S in Ss) for k in keys}


def sweep_sums(S):
    """Score of a 12-TR sweep of k starting at t0 (cut at the end of the run)."""
    return {k: np.array([sum(S[k][t0 + i, i] for i in range(12) if t0 + i < T) for t0 in range(T)]) for k in keys}


In [7]:
def merge_blanks(chain):
    out = []
    for e in chain:
        if out and e[0] == 'b' and out[-1][0] == 'b':
            out[-1] = ('b', out[-1][1], out[-1][2] + e[2])
        else:
            out.append(e)
    return out


def dp_free(S):
    """Exact best chain, no constraint: sweeps (8 directions, 12 TR) and blanks (1-14 TR, score 0).
    Returns chain [(k, t0, L)], total score, and for each sweep the global margin over the best chain with another
    direction at the same place (max-marginals)."""
    sc = sweep_sums(S)
    fwd = np.full(T + 1, -np.inf); fwd[0] = 0; back = [None] * (T + 1)
    for t in range(T):
        if not np.isfinite(fwd[t]): continue
        for b in range(1, 15):
            u = min(T, t + b)
            if fwd[t] > fwd[u]: fwd[u] = fwd[t]; back[u] = (t, 'b', u - t)
        for k in keys:
            u = min(T, t + 12); v = fwd[t] + sc[k][t]
            if v > fwd[u]: fwd[u] = v; back[u] = (t, k, 12)
    bwd = np.full(T + 1, -np.inf); bwd[T] = 0
    for t in range(T - 1, -1, -1):
        bwd[t] = max(max(bwd[min(T, t + b)] for b in range(1, 15)), max(sc[k][t] + bwd[min(T, t + 12)] for k in keys))
    chain, u = [], T
    while u > 0:
        t, k, l = back[u]; chain.append((k, t, l)); u = t
    chain = merge_blanks(chain[::-1])
    opt = fwd[T]
    conf = []
    for k, t0, l in chain:
        if k == 'b': continue
        alt = max(fwd[t0] + sc[k2][t0] + bwd[min(T, t0 + 12)] for k2 in keys if k2 != k)
        conf.append((k, t0, opt - alt))
    return chain, opt, conf


def dp_perm(S, init_blank=range(3, 8), ins_blank=(5, 6)):
    """Exact best chain with the generator structure seen in A: initial blank, the 8 directions once each with one
    inserted half-cycle blank, then free sweeps until the end (at least 7 directions completed)."""
    sc = sweep_sums(S)
    full = (1 << len(keys)) - 1

    @lru_cache(maxsize=None)
    def V(t, mask, bl, pb):
        if t >= T:
            return (0.0 if bin(mask).count('1') >= 7 else -np.inf), None
        best = (-np.inf, None)
        for i, k in enumerate(keys):
            if mask == full or not (mask >> i) & 1:
                nm = mask if mask == full else mask | (1 << i)
                v = sc[k][t] + V(min(T, t + 12), nm, bl, False)[0]
                if v > best[0]: best = (v, (k, 12, nm, bl, False))
        if mask != full and not bl and mask and not pb:
            for b in ins_blank:
                v = V(min(T, t + b), mask, True, True)[0]
                if v > best[0]: best = (v, ('b', b, mask, True, True))
        return best

    starts = [(V(l, 0, False, True)[0], l) for l in init_blank]
    opt, l0 = max(starts)
    chain, t, state = [('b', 0, l0)], l0, (0, False, True)
    while t < T:
        _, ch = V(t, *state)
        k, l, nm, bl, pb = ch
        chain.append((k, t, min(l, T - t) if k == 'b' else 12))
        t, state = t + l, (nm, bl, pb)
    V.cache_clear()
    return chain, opt


def sweeps_of(chain):
    return [(k, t0) for k, t0, l in chain if k != 'b' and t0 + 12 <= T]


def n_correct(chain, true):
    """Number of full sweeps of the true chain found in the chain (same direction, start +-1 TR) / number of true sweeps."""
    est = sweeps_of(chain)
    tr = sweeps_of(true)
    ok = sum(any(k == k2 and abs(t - t2) <= 1 for k2, t2 in est) for k, t in tr)
    return f'{ok}/{len(tr)}'


def agreement(c1, c2):
    a, b = sweeps_of(c1), sweeps_of(c2)
    ok = sum(any(k == k2 and abs(t - t2) <= 1 for k2, t2 in b) for k, t in a)
    return f'{ok}/{len(a)}'


def cstr(chain):
    return ' '.join(f'b{l}' if k == 'b' else k for k, t0, l in chain)


In [8]:
calib = []   # (case, direction, start, margin, correct) for the sweeps of the validation runs
vert_cache = {}
def decode(s, r, train):
    """train: list of (run, stim (50,50,120)). Returns decoded movie, gain, mu, LORO frame corr."""
    if s not in vert_cache: vert_cache[s] = select_vertices(s, dec_rois, dec_nmax)
    idx = vert_cache[s]
    tr = [(features(load_run(s, rr)[:, idx]), ds(st)) for rr, st in train]
    pred, a, gain, loro, mu = train_decoder(tr)
    return pred(features(load_run(s, r)[:, idx])), gain, mu, loro


def analyse(s, r, train, true=None, label=''):
    D, gain, mu, loro = decode(s, r, train)
    Sd = normalise(decoder_scores(D, gain, mu), center=False)
    Sm = normalise(model_scores(s, r), center=True)
    Sc = combine(Sd, Sm)
    res = {'case': label, 'subject': s, 'run': r, 'LORO frame corr (train)': loro}
    chains = {}
    for name, S in [('decoder', Sd), ('pRF model', Sm), ('combined', Sc)]:
        ch, opt, conf = dp_free(S)
        chains[name] = ch
        res[f'{name}: chain'] = cstr(ch)
        if name == 'combined':
            res['combined: min margin'] = min(c_[2] for c_ in conf) if conf else np.nan
            res['combined: margins'] = ' '.join(f'{k}@{t0}:{m:.1f}' for k, t0, m in conf)
            res['_conf'] = conf
            if true is not None:
                tr = sweeps_of(true)
                for k, t0, m in conf:
                    if t0 + 12 <= T:
                        calib.append((label, k, t0, m, any(k == k2 and abs(t0 - t2) <= 1 for k2, t2 in tr)))
        if true is not None:
            res[f'{name}: correct'] = n_correct(ch, true)
    chp, optp = dp_perm(Sc)
    chains['combined + permutation'] = chp
    res['combined + permutation: chain'] = cstr(chp)
    res['free vs permutation (agreement)'] = agreement(chains['combined'], chp)
    if true is not None:
        res['combined + permutation: correct'] = n_correct(chp, true)
        res['frame corr decoded vs true'] = frame_corr(D, ds(chain_to_stim(true)))
    res['pRF score combined'] = prf_score(chain_to_stim(chains['combined']), s, r)
    res['pRF score combined + permutation'] = prf_score(chain_to_stim(chp), s, r)
    return res, chains, D


def show(rows, cols=None):
    df = pd.DataFrame([{k: v for k, v in r_.items() if not k.startswith('_')} for r_ in rows]).set_index('case')
    pd.set_option('display.max_colwidth', 120)
    print(df[cols].T.to_string() if cols else df.T.to_string())
    return df

stim_of = lambda r: run_stim(r)                    # known stimuli of rings / wedges


In [9]:
def chain_from_vdm(stim):
    return [(k, s, l) for k, s, l in segment(stim)]


def fixed_from(c1, c2):
    """Sweeps on which two chains agree (same direction, start +-1 TR); time taken from c2."""
    b = sweeps_of(c2)
    return [(k, t2) for k, t in sweeps_of(c1) for k2, t2 in b if k == k2 and abs(t - t2) <= 1]


def enumerate_chains(fixed, max_n=50000, init_blank=range(3, 8), ins_blank=(5, 6)):
    """All chains with the generator structure (initial blank, 8 distinct directions with one inserted half blank,
    then free sweeps) that contain every fixed sweep (direction, start +-1 TR)."""
    fixed = sorted(fixed, key=lambda z: z[1])
    out = []

    def rec(t, used, bl, pb, chain, nfix):
        if len(out) >= max_n: return
        # prune: a fixed sweep whose window has passed without being placed
        if nfix < len(fixed) and t > fixed[nfix][1] + 1: return
        if t >= T:
            if nfix == len(fixed) and len(used) >= 7: out.append(chain)
            return
        must = fixed[nfix] if (nfix < len(fixed) and abs(t - fixed[nfix][1]) <= 1) else None
        full = len(used) == 8
        for k in keys:
            if must is not None and k != must[0]: continue
            if not full and k in used: continue
            rec(t + 12, used if full else used | {k}, bl, False, chain + [(k, t, 12)], nfix + (must is not None))
        if must is None and not full and not bl and used and not pb:
            for b in ins_blank:
                rec(t + b, used, True, True, chain + [('b', t, min(b, T - t))], nfix)

    for l in init_blank:
        rec(l, frozenset(), False, True, [('b', 0, l)], 0)
    return out


In [10]:
class RunScorer:
    """Scores of a candidate chain for one run:
    S1 combined frame evidence (decoder + pRF patterns, additive), S2 pRF forward model (median correlation over vertices),
    S3 global signal (mean of V1-V3 vertices: blanks vs sweeps), S4 travelling-wave latency (order in which vertices
    respond within each sweep vs order in which the bar covers their pRF centre)."""

    def __init__(self, s, r, Sc):
        self.s, self.r = s, r
        self.sweep_sc = sweep_sums(Sc)
        roi, prf = load_subject(s)
        Dp = prf['LeftEye' if 'LeftEye' in r else 'RightEye']
        y = load_run(s, r)
        idx = np.where((Dp[0] > 0.2) & (Dp[1] < 6) & np.isin(roi, [1, 2, 3]) & np.isfinite(y).all(0) & (y.std(0) > 0))[0]
        x0, y0, sd, n = Dp[7][idx], Dp[8][idx], np.clip(Dp[4][idx], 0.2, None), np.clip(Dp[11][idx], 0.05, 1)
        G = np.exp(-((Xd.ravel()[:, None] - x0) ** 2 + (Yd.ravel()[:, None] - y0) ** 2) / (2 * sd ** 2))
        self.drv = {k: np.clip(np.moveaxis(tmpl(k, 12), -1, 0).reshape(12, -1) @ G, 0, None) ** n for k in keys}
        self.Y = zc(y[:, idx]).astype(np.float32)
        self.gY = zc(y[:, idx].mean(1)[:, None])[:, 0]
        # travelling-wave latency table
        good = Dp[0][idx] > 0.3
        ii = np.clip(np.round(24.5 - y0 / pix).astype(int), 0, 49)
        jj = np.clip(np.round(x0 / pix + 24.5).astype(int), 0, 49)
        self.lat = {}
        for k in keys:
            fr = tmpl(k, 12)[ii, jj, :] > 0                    # (nv, 12) frames covering each pRF centre
            cov = fr.sum(1) > 0
            cover_t = np.where(cov, (fr * np.arange(12)).sum(1) / np.maximum(fr.sum(1), 1), np.nan)
            v = good & cov
            tab = np.zeros(T)
            for t0 in range(T - 4):
                seg = self.Y[t0 + 2: t0 + 14, v]
                if len(seg) < 6: continue
                w = seg - seg.min(0)
                com = (w * np.arange(len(seg))[:, None]).sum(0) / (w.sum(0) + 1e-9)
                tab[t0] = np.corrcoef(com, cover_t[v])[0, 1]
            self.lat[k] = np.nan_to_num(tab)

    def scores(self, chain):
        s1 = sum(self.sweep_sc[k][t0] for k, t0, l in chain if k != 'b')
        s4 = sum(self.lat[k][t0] for k, t0, l in chain if k != 'b' and t0 < T)
        d = np.zeros_like(self.Y)
        for k, t0, l in chain:
            if k == 'b' or t0 >= T: continue
            n = min(12, T - t0)
            d[t0:t0 + n] = self.drv[k][:n]
        p = conv_hrf(d)
        best2, best3 = -np.inf, -np.inf
        gp = p.mean(1)
        for sh in range(-2, 3):
            q = np.roll(p, sh, 0); g = np.roll(gp, sh)
            if sh > 0: q[:sh] = p[0]; g[:sh] = gp[0]
            elif sh < 0: q[sh:] = p[-1]; g[sh:] = gp[-1]
            best2 = max(best2, float(np.median((zc(q) * self.Y).mean(0))))
            best3 = max(best3, float(np.corrcoef(g, self.gY)[0, 1]))
        return {'S1 frame evidence': s1, 'S2 pRF model': best2, 'S3 global signal': best3, 'S4 latency': s4}


crit = ['S1 frame evidence', 'S2 pRF model', 'S3 global signal', 'S4 latency']


def rank_hypotheses(scorer, hyps):
    df = pd.DataFrame([scorer.scores(h) for h in hyps])
    for c_ in crit:
        df[c_ + ' (z)'] = (df[c_] - df[c_].mean()) / (df[c_].std() + 1e-12)
    df['all (sum z)'] = df[[c_ + ' (z)' for c_ in crit]].sum(1)
    df['chain'] = [cstr(h) for h in hyps]
    return df


In [11]:
CR = ['S1 frame evidence', 'S2 pRF model', 'S4 latency']


def slot_test(scorer, chain):
    """For each full sweep of the chain: the 8 directions are tried at its place (everything else fixed, repeats allowed)."""
    out = []
    for j, (k, t0, l) in enumerate(chain):
        if k == 'b' or t0 + 12 > T: continue
        alts = []
        for k2 in keys:
            ch = list(chain); ch[j] = (k2, t0, l)
            alts.append(scorer.scores(ch))
        df = pd.DataFrame(alts, index=keys)
        rec = {'slot': j, 'start': t0, 'given': k}
        z = 0
        for c_ in CR:
            s_ = df[c_].sort_values(ascending=False)
            rec[f'{c_}: best'] = s_.index[0]
            rec[f'{c_}: margin'] = s_.iloc[0] - s_.iloc[1]
            rec[f'{c_}: given - best other'] = df.loc[k, c_] - df[c_].drop(k).max()
            z = z + (df[c_] - df[c_].mean()) / (df[c_].std() + 1e-12)
        rec['sum z: best'] = z.idxmax()
        rec['sum z: given - best other'] = z[k] - z.drop(k).max()
        out.append(rec)
    return pd.DataFrame(out)


def triple_test(scorer, chain, windows=((0, 3), (3, 6), (6, 9))):
    """Hide 3 consecutive sweeps and try every combination of directions (8^3 = 512, repeats allowed)."""
    sw = [j for j, (k, t0, l) in enumerate(chain) if k != 'b' and t0 + 12 <= T]
    rows_ = []
    for a, b in windows:
        hid = sw[a:b]
        if len(hid) < 3: continue
        hyps, is_true = [], []
        for combo in itertools.product(keys, repeat=len(hid)):
            ch = list(chain)
            for j, k2 in zip(hid, combo):
                ch[j] = (k2, chain[j][1], chain[j][2])
            hyps.append(ch); is_true.append(all(chain[j][0] == k2 for j, k2 in zip(hid, combo)))
        df = pd.DataFrame([scorer.scores(h) for h in hyps])
        df['sum z'] = sum((df[c_] - df[c_].mean()) / (df[c_].std() + 1e-12) for c_ in CR)
        it = int(np.where(is_true)[0][0])
        rec = {'hidden sweeps': f'{a + 1}-{b}'}
        for c_ in CR + ['sum z']:
            rec[f'rank of truth: {c_}'] = int(df[c_].rank(ascending=False, method='min')[it])
            rec[f'winner: {c_}'] = ' '.join(hyps[int(df[c_].idxmax())][j][0] for j in hid)
        rec['truth'] = ' '.join(chain[j][0] for j in hid)
        rows_.append(rec)
    return pd.DataFrame(rows_)


## Step 1 — validation with repeats allowed

In [12]:
val_cases = [
    ('sub-16', 'BarsLeftEye_run-02', 'A', [(r, run_stim(r)) for r in RW]),
    ('sub-13', 'BarsLeftEye_run-01', 'A', [(r, run_stim(r)) for r in RW]),
    ('sub-13', 'BarsRightEye_run-02', 'A', [(r, run_stim(r)) for r in RW]),
    ('sub-05', 'BarsRightEye_run-01', 'B', [(r, run_stim(r)) for r in RW] +
                                       [(r, A) for r in ['BarsLeftEye_run-01', 'BarsLeftEye_run-02', 'BarsRightEye_run-02']]),
    ('sub-17', 'BarsLeftEye_run-01', 'B', [(r, run_stim(r)) for r in RW]),
    ('sub-14', 'BarsLeftEye_run-01', 'B', [(r, run_stim(r)) for r in RW]),
]
slot_rows, triple_rows = [], []
for s, r, kind, train in val_cases:
    run_cache.clear()                                   # keep memory low
    try:
        D, gain, mu, _ = decode(s, r, train)
    except Exception as e:
        print(s, r, 'skipped:', e); continue
    Sc = combine(normalise(decoder_scores(D, gain, mu), center=False), normalise(model_scores(s, r), center=True))
    sc_ = RunScorer(s, r, Sc)
    st = slot_test(sc_, true_chain[kind]); st.insert(0, 'run', f'{s} {r} ({kind})')
    slot_rows.append(st)
    tt = triple_test(sc_, true_chain[kind]); tt.insert(0, 'run', f'{s} {r} ({kind})')
    triple_rows.append(tt)
    print(s, r, 'done', flush=True)
slots = pd.concat(slot_rows, ignore_index=True)
triples = pd.concat(triple_rows, ignore_index=True)

print('=== Single sweep test (8 directions tried at each sweep of known runs, repeats allowed)')
acc = {c_: (slots[f'{c_}: best'] == slots['given']).mean() for c_ in CR + ['sum z']}
print('accuracy:', {k: f'{v:.0%}' for k, v in acc.items()}, '| n sweeps =', len(slots))
for c_ in CR + ['sum z']:
    wrong = slots[slots[f'{c_}: best'] != slots['given']]
    conf = (wrong['given'] + ' -> ' + wrong[f'{c_}: best']).value_counts()
    print(f'\n{c_}: {len(wrong)} errors', dict(conf))
thr_slot = {c_: -slots.loc[slots[f'{c_}: best'] != slots['given'], f'{c_}: given - best other'].min()
            if (slots[f'{c_}: best'] != slots['given']).any() else 0 for c_ in CR + ['sum z']}
print('\nLargest advantage of a wrong direction over the truth (threshold):', {k: round(v, 3) for k, v in thr_slot.items()})
print('\n=== Triple test (3 hidden sweeps, 512 combinations with repeats)')
print(triples.to_string())
print('\nTruth ranked 1st:', {c_: int((triples[f'rank of truth: {c_}'] == 1).sum()) for c_ in CR + ['sum z']}, 'out of', len(triples))


sub-16 BarsLeftEye_run-02 done
sub-13 BarsLeftEye_run-01 done
sub-13 BarsRightEye_run-02 done
sub-05 BarsRightEye_run-01 done
sub-17 BarsLeftEye_run-01 done
sub-14 BarsLeftEye_run-01 done
=== Single sweep test (8 directions tried at each sweep of known runs, repeats allowed)
accuracy: {'S1 frame evidence': '82%', 'S2 pRF model': '95%', 'S4 latency': '89%', 'sum z': '93%'} | n sweeps = 57

S1 frame evidence: 10 errors {'45- -> 0-': 5, '45+ -> 0+': 4, '135- -> 0+': 1}

S2 pRF model: 3 errors {'45- -> 0-': 2, '45+ -> 0+': 1}

S4 latency: 6 errors {'45+ -> 0+': 2, '45- -> 0-': 1, '90+ -> 45+': 1, '90- -> 45-': 1, '0+ -> 135-': 1}

sum z: 4 errors {'45- -> 0-': 3, '45+ -> 0+': 1}

Largest advantage of a wrong direction over the truth (threshold): {'S1 frame evidence': 5.268, 'S2 pRF model': 0.003, 'S4 latency': 0.041, 'sum z': 0.175}

=== Triple test (3 hidden sweeps, 512 combinations with repeats)
                               run hidden sweeps  rank of truth: S1 frame evidence winner: S1

## Step 2 — suspect runs: is each sweep (including the repeated ones) confirmed against the 7 other directions?

In [13]:
run_cache.clear()
comb_dir = '/Users/uriel/Desktop/fig_od/sub16_combined'
sus_slots = {}
for r in SUSPECTS:
    train = [(rr, run_stim(rr)) for rr in RW] + [('BarsLeftEye_run-02', A)] + \
            [(o, np.load(f'{comb_dir}/sub-16_{o}_combined_vdm.npy')) for o in SUSPECTS if o != r]
    D, gain, mu, _ = decode('sub-16', r, train)
    Sc = combine(normalise(decoder_scores(D, gain, mu), center=False), normalise(model_scores('sub-16', r), center=True))
    sc_ = RunScorer('sub-16', r, Sc)
    for variant in ['combined', 'combined_permutation']:
        ch = chain_from_vdm(np.load(f'{comb_dir}/sub-16_{r}_{variant}_vdm.npy'))
        st = slot_test(sc_, ch)
        dirs = [k for k, t0, l in ch if k != 'b' and t0 + 12 <= T]
        st['repeated in chain'] = [dirs.count(k) > 1 for k in st['given']]
        sus_slots[(r, variant)] = st
        print(f'\n=== {r} | {"free" if variant == "combined" else "with structure"} chain: {cstr(ch)}')
        cols = ['start', 'given', 'repeated in chain'] + [f'{c_}: best' for c_ in CR] + ['sum z: best', 'S4 latency: given - best other',
                                                                                         'S2 pRF model: given - best other']
        print(st[cols].round(3).to_string())
        n_ok = {c_: int((st[f'{c_}: best'] == st['given']).sum()) for c_ in CR + ['sum z']}
        print('sweeps confirmed (best = given):', n_ok, 'out of', len(st))



=== BarsLeftEye_run-01 | free chain: b5 90+ 0+ b5 0+ b1 135- 45- 135+ 90- 135- 0- b1
   start given  repeated in chain S1 frame evidence: best S2 pRF model: best S4 latency: best sum z: best  S4 latency: given - best other  S2 pRF model: given - best other
0      5   90+              False                     90+                90+              90+         90+                           0.072                             0.009
1     17    0+               True                      0+                 0+               0+          0+                           0.057                             0.005
2     34    0+               True                      0+                 0+              45+          0+                          -0.059                             0.003
3     47  135-               True                    135-               135-             135-        135-                           0.168                             0.010
4     59   45-              False                     

## Step 3 — OD of sub-16 with each design

In [14]:
# OD of sub-16 with each candidate design (same principle as make_od_img.py: common regressor = mean of the pRF predictions of
# the two eyes, z-scored over the concatenation, mean removed per run; beta = sum(xy)/sum(x2) per eye; OD = (bL - bR)/(bL + bR)
# where both betas > 0; weighted median, weights = mean pRF R2 of the two eyes). pRFs: RingsWedges fits (independent of the bars).
s = 'sub-16'
roi, prf = load_subject(s)
r2m = (prf['LeftEye'][0] + prf['RightEye'][0]) / 2
vid = np.where(np.isin(roi, [1, 2, 3]) & np.isfinite(r2m) & (r2m > 0.1))[0]
print(len(vid), 'vertices')


def eye_pred(stim, eye, idx):
    Dp = prf[eye]
    x0, y0, sd, n = Dp[7][idx], Dp[8][idx], np.clip(Dp[4][idx], 0.2, None), np.clip(Dp[11][idx], 0.05, 1)
    out = np.zeros((120, len(idx)), np.float32)
    for c0 in range(0, len(idx), 4000):
        sl = slice(c0, c0 + 4000)
        G = np.exp(-((Xd.ravel()[:, None] - x0[sl]) ** 2 + (Yd.ravel()[:, None] - y0[sl]) ** 2) / (2 * sd[sl] ** 2))
        out[:, sl] = conv_hrf(np.clip(np.moveaxis(stim, -1, 0).reshape(120, -1) @ G, 0, None) ** n[sl])
    return out


def od_for(design):
    """design: dict run -> stim (50,50,120)."""
    runs = list(design)
    X = np.vstack([(eye_pred(design[r], 'LeftEye', vid) + eye_pred(design[r], 'RightEye', vid)) / 2 for r in runs])
    X = (X - X.mean(0)) / (X.std(0) + 1e-9)
    num = {'LeftEye': 0, 'RightEye': 0}; den = {'LeftEye': 0, 'RightEye': 0}
    for i, r in enumerate(runs):
        x = X[i * 120:(i + 1) * 120]; x = x - x.mean(0)
        y = load_run(s, r)[:, vid]; y = y - y.mean(0)
        e = 'LeftEye' if 'LeftEye' in r else 'RightEye'
        num[e] = num[e] + (x * y).sum(0); den[e] = den[e] + (x * x).sum(0)
    bL, bR = num['LeftEye'] / den['LeftEye'], num['RightEye'] / den['RightEye']
    ok = (bL > 0) & (bR > 0)
    od = np.where(ok, (bL - bR) / np.where(ok, bL + bR, 1), np.nan)
    return od


def wmedian(v, w):
    m = np.isfinite(v); v, w = v[m], w[m]
    o = np.argsort(v); cw = np.cumsum(w[o])
    return float(v[o][np.searchsorted(cw, cw[-1] / 2)])

base = {r_: run_stim(r_) for r_ in RW}
designs = {
    'exclude the 3 suspect runs': {**base, 'BarsLeftEye_run-02': A},
    'current analysis (A for all bars runs)': {**base, 'BarsLeftEye_run-02': A, **{r_: A for r_ in SUSPECTS}},
    'recovered: free chains': {**base, 'BarsLeftEye_run-02': A,
                               **{r_: np.load(f'{comb_dir}/sub-16_{r_}_combined_vdm.npy') for r_ in SUSPECTS}},
    'recovered: chains with structure': {**base, 'BarsLeftEye_run-02': A,
                                         **{r_: np.load(f'{comb_dir}/sub-16_{r_}_combined_permutation_vdm.npy') for r_ in SUSPECTS}},
}
ods = {name: od_for(d) for name, d in designs.items()}
rows_od = []
w = r2m[vid]
for name, od in ods.items():
    rec = {'design': name}
    for rv, rn in [(1, 'V1'), (2, 'V2'), (3, 'V3')]:
        m = roi[vid] == rv
        rec[f'{rn} OD (weighted median)'] = wmedian(od[m], w[m])
        rec[f'{rn} % LE'] = float(np.nanmean(od[m] > 0) * 100)
        rec[f'{rn} n valid'] = int(np.isfinite(od[m]).sum())
    ref = ods['exclude the 3 suspect runs']
    m = np.isfinite(od) & np.isfinite(ref)
    rec['vertex-wise r with "exclude"'] = float(np.corrcoef(od[m], ref[m])[0, 1])
    rows_od.append(rec)
od_tab = pd.DataFrame(rows_od).set_index('design')
print(od_tab.round(3).T.to_string())


15489 vertices
design                        exclude the 3 suspect runs  current analysis (A for all bars runs)  recovered: free chains  recovered: chains with structure
V1 OD (weighted median)                            0.004                                   0.140                   0.039                             0.040
V1 % LE                                           51.824                                  72.486                  58.296                            57.643
V1 n valid                                      4624.000                                4610.000                4624.000                          4612.000
V2 OD (weighted median)                            0.018                                   0.169                   0.054                             0.051
V2 % LE                                           51.661                                  73.754                  57.297                            57.046
V2 n valid                                      3924.00

## How to read

- **Step 1**: if S4 / S2 / sum z keep ~100 % accuracy and never make the 45+ -> 0+ error, they are not fooled by repeats. Their verdict on the suspect runs can then be trusted.
- **Step 2**: for each repeated sweep (0+, 135-) of the free chains, whether the validated criteria confirm it against the 7 other directions, with a margin (given - best other) above the validation threshold.
  - If yes, the pilot really repeated directions: use the free chains.
  - If the criteria prefer 45+ there, the free chain made the known confusion: use the corrected chain (with structure).
- **Step 3**: the OD of sub-16 per ROI for each design. Compare the recovered designs with "exclude" (reference without assumption). The line "current analysis (A everywhere)" shows how much the error currently biases sub-16.

In [15]:
# Corrected chains: free chain with the contested 0+ replaced by 45+ (one per run) -> 8 directions + a repeated 135-, like A
fix = {'BarsLeftEye_run-01': 34, 'BarsRightEye_run-01': 94, 'BarsRightEye_run-02': 29}
final_dir = '/Users/uriel/Desktop/fig_od/sub16_final'
os.makedirs(final_dir, exist_ok=True)
cands = {}
for r in SUSPECTS:
    free = chain_from_vdm(np.load(f'{comb_dir}/sub-16_{r}_combined_vdm.npy'))
    corr = [('45+', t0, l) if (k == '0+' and abs(t0 - fix[r]) <= 1) else (k, t0, l) for k, t0, l in free]
    cands[r] = {'free': free, 'structure': chain_from_vdm(np.load(f'{comb_dir}/sub-16_{r}_combined_permutation_vdm.npy')),
                'corrected': corr}
    dirs = [k for k, t0, l in corr if k != 'b' and t0 + 12 <= T]
    print(f'{r}: {cstr(corr)} | distinct = {len(set(dirs))} | repeated = {[k for k in set(dirs) if dirs.count(k) > 1]}')
dA = [k for k, t0, l in true_chain['A'] if k != 'b' and t0 + 12 <= T]
print(f"A: {cstr(true_chain['A'])} | distinct = {len(set(dA))} | repeated = {[k for k in set(dA) if dA.count(k) > 1]}")

# 1) scores of the 3 candidates (same scorer as step 2) + single-sweep test of the corrected chain
for r in SUSPECTS:
    train = [(rr, run_stim(rr)) for rr in RW] + [('BarsLeftEye_run-02', A)] + \
            [(o, chain_to_stim(cands[o]['corrected'])) for o in SUSPECTS if o != r]
    D, gain, mu, _ = decode('sub-16', r, train)
    Sc = combine(normalise(decoder_scores(D, gain, mu), center=False), normalise(model_scores('sub-16', r), center=True))
    sc_ = RunScorer('sub-16', r, Sc)
    print(f'\n=== {r}')
    print(pd.DataFrame({n: sc_.scores(ch) for n, ch in cands[r].items()}).round(4).to_string())
    st = slot_test(sc_, cands[r]['corrected'])
    print(st[['start', 'given'] + [f'{c_}: best' for c_ in CR] + ['sum z: best', 'S2 pRF model: given - best other',
                                                                 'S4 latency: given - best other']].round(3).to_string())

# 2) round trip: LE02 decoded from the 3 suspect runs (never sees LE02)
rt_ = []
for n in ['free', 'structure', 'corrected']:
    train = [(rr, run_stim(rr)) for rr in RW] + [(o, chain_to_stim(cands[o][n])) for o in SUSPECTS]
    D, gain, mu, _ = decode('sub-16', 'BarsLeftEye_run-02', train)
    ch, _, _ = dp_free(normalise(decoder_scores(D, gain, mu), center=False))
    rt_.append({'chains': n, 'LE02 frame corr vs A': frame_corr(D, ds(A)), 'LE02 sweeps correct': n_correct(ch, true_chain['A']),
                'decoded LE02': cstr(ch)})
print('\nRound trip'); print(pd.DataFrame(rt_).round(4).to_string())

# 3) OD with the corrected chains
designs['recovered: corrected chains'] = {**base, 'BarsLeftEye_run-02': A, **{r_: chain_to_stim(cands[r_]['corrected']) for r_ in SUSPECTS}}
ods['recovered: corrected chains'] = od_for(designs['recovered: corrected chains'])
rows_od = []
for name, od in ods.items():
    rec = {'design': name}
    for rv, rn in [(1, 'V1'), (2, 'V2'), (3, 'V3')]:
        m = roi[vid] == rv
        rec[f'{rn} OD (weighted median)'] = wmedian(od[m], w[m]); rec[f'{rn} % LE'] = float(np.nanmean(od[m] > 0) * 100)
    ref = ods['exclude the 3 suspect runs']; m = np.isfinite(od) & np.isfinite(ref)
    rec['vertex-wise r with "exclude"'] = float(np.corrcoef(od[m], ref[m])[0, 1])
    rows_od.append(rec)
print(); print(pd.DataFrame(rows_od).set_index('design').round(3).T.to_string())

for r in SUSPECTS:
    np.save(f'{final_dir}/sub-16_{r}_vdm.npy', chain_to_stim(cands[r]['corrected']))
print('\nSaved in', final_dir)

BarsLeftEye_run-01: b5 90+ 0+ b5 45+ b1 135- 45- 135+ 90- 135- 0- b1 | distinct = 8 | repeated = ['135-']
BarsRightEye_run-01: b5 135- 0+ 135- 45- 135+ 90+ b5 0- 45+ b1 90- b1 | distinct = 8 | repeated = ['135-']
BarsRightEye_run-02: b5 135- 0- 45+ 135+ 45- b6 135- 90+ 90- 0+ b1 | distinct = 8 | repeated = ['135-']
A: b6 135+ 90+ b5 45- 0+ 135- 0- 45+ 90- 135- 135- | distinct = 8 | repeated = ['135-']

=== BarsLeftEye_run-01
                       free  structure  corrected
S1 frame evidence  203.7716   165.0881   210.0242
S2 pRF model         0.3632     0.3206     0.3606
S3 global signal     0.3917     0.3881     0.3777
S4 latency           5.2678     4.8040     5.3272
   start given S1 frame evidence: best S2 pRF model: best S4 latency: best sum z: best  S2 pRF model: given - best other  S4 latency: given - best other
0      5   90+                     90+                90+              90+         90+                             0.008                           0.072
1     17    0+ 